# Random Forest Classification and Feature Importance

## Project Introduction

The goal of this project is to train a Random Forest classification model using the Breast Cancer Wisconsin dataset.

The project aims to predict the target class based on the available features and evaluate the model's performance.

Feature importance is also used to identify which features contribute most to the model's tree-based decisions.

This project helps us understand how Random Forest works and how feature importance can be used to interpret a machine learning model.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    recall_score
)


## 1. Import Libraries

In this step, we import the libraries required for data handling, visualization, model training, and evaluation.

* **Pandas:** Used to organize and manipulate data in DataFrames.
* **Matplotlib:** Used to create charts and visualize feature importance.
* **load_breast_cancer:** Used to load the Breast Cancer Wisconsin dataset.
* **train_test_split:** Used to divide the dataset into training and testing sets.
* **RandomForestClassifier:** Used to create and train the Random Forest classification model.
* **accuracy_score:** Used to calculate the percentage of correct predictions.
* **classification_report:** Used to evaluate precision, recall, and F1-score.
* **confusion_matrix:** Used to show correct and incorrect predictions for each class.


In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data , columns=data.feature_names)
y = pd.Series(data.target , name="target")

X.head()


In [ ]:
X.tail()

In [ ]:
y.head(10)

In [ ]:
y.tail(10)

In [ ]:
X.info()

In [ ]:
X.shape

In [ ]:
y.value_counts()

In [ ]:
X.isnull().sum()

## 2. Load and Explore the Dataset

In this step, we load the Breast Cancer Wisconsin dataset and explore its structure before training the model.

* **load_breast_cancer():** Loads the dataset provided by scikit-learn.
* **X:** Contains the input features used by the model to make predictions.
* **y:** Contains the target labels that the model must predict.
* **pd.DataFrame():** Organizes the feature data into a table with named columns.
* **pd.Series():** Stores the target labels in a one-dimensional structure.
* **head():** Displays the first five rows to inspect the data.
* **info():** Displays information about the dataset, including column names, non-null values, data types, and memory usage. It helps us understand the dataset structure and identify columns that may contain missing values.
* **shape:** Shows the number of rows and columns.
* **value_counts():** Shows how many samples belong to each target class.
* **isnull().sum():** Counts the total number of missing values in X.

Exploring the dataset helps us understand the available features and check whether any data preparation is needed before training.


In [ ]:
std = StandardScaler()
X_std = std.fit_transform(X)

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(
    X_std,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 3. Splitting the Dataset

We split the dataset into training and testing sets to evaluate the model on data it has not seen during training.

* **X_train:** The input features used for training.
* **X_test:** The input features used for testing.
* **y_train:** The correct target labels for the training data.
* **y_test:** The correct target labels for the testing data.
* **test_size=0.2:** Allocates 20% of the data for testing and 80% for training.
* **random_state=42:** Makes the random split reproducible.
* **stratify=y:** Helps preserve the proportion of each target class in both sets.

This split helps us evaluate whether the model can generalize to unseen data rather than only memorizing the training examples.


In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_prec = model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_prec)

print("Training Accuracy =", train_acc)
print("Test Accuracy =", test_acc)

print("\\nTraining class counts:")
print(y_train.value_counts().sort_index())

print("\\nClassification Report:")
print(classification_report(
    y_test,
    y_prec,
    target_names=data.target_names
))

print("Confusion Matrix (rows = true labels, columns = predicted labels):")
print(confusion_matrix(y_test, y_prec, labels=[0, 1]))


## 4. Building and Training the Random Forest Model

We use Random Forest because it combines multiple decision trees to make classification predictions. Combining many trees can improve prediction stability compared with relying on a single tree.

### Model Parameters

* **n_estimators=200:** Creates 200 decision trees. More trees can stabilize predictions, but they also increase computation.
* **max_depth=10:** Limits each tree's maximum depth. We will compare this value with other depths rather than assume it is automatically the best choice.
* **random_state=42:** Makes the model's random processes reproducible.
* **class_weight="balanced":** Automatically adjusts class weights according to the class frequencies in the training data. The training class counts printed above show the distribution used by the model.

The **fit()** method trains the model using `X_train` and `y_train`. We report both training and test accuracy: a much higher training score than test score can indicate overfitting. Accuracy alone is not enough for this medical dataset, so malignant-class recall is also considered.

**Note:** `StandardScaler` is not required by Random Forest because tree splits do not depend on feature scale. In this notebook it is retained to match the original workflow; for a leakage-safe workflow, split first and fit the scaler on training data only (or omit scaling for Random Forest).


## 5. Hyperparameter Comparison and Cross-Validation

A single train/test split provides one estimate of performance, and the test set contains only 114 samples. To make the comparison more reliable, we use **5-fold stratified cross-validation** on the training set. Each fold preserves the class proportions as closely as possible.

We compare `n_estimators` values of 100, 200, and 300, and `max_depth` values of 5, 10, and `None`. The table reports mean cross-validation accuracy and its standard deviation, plus training accuracy, test accuracy, and malignant-class recall.

* **Cross-validation accuracy:** Mean accuracy across the five validation folds.
* **Training accuracy:** Performance on the samples used to fit that candidate model.
* **Test accuracy:** Performance on the held-out test set.
* **Malignant recall:** Fraction of actual malignant cases correctly identified (`pos_label=0`).

A large gap between training accuracy and cross-validation/test accuracy suggests overfitting. `max_depth=10` is justified only if the comparison shows a useful balance between validation performance and the training-to-test gap. Choose hyperparameters based primarily on cross-validation results; use the test set for final evaluation rather than repeatedly tuning against it.

`class_weight="balanced"` uses the training class counts to give more weight to the less frequent class. Check the printed counts: class 0 is malignant and class 1 is benign in this dataset. This setting is reasonable to evaluate when missing malignant cases is especially costly, but the recall results must confirm whether it helps.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for n in [100, 200, 300]:
    for depth in [5, 10, None]:
        candidate = RandomForestClassifier(
            n_estimators=n,
            max_depth=depth,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )

        scores = cross_val_score(
            candidate,
            X_train,
            y_train,
            cv=cv,
            scoring="accuracy",
            n_jobs=-1
        )

        candidate.fit(X_train, y_train)
        train_score = accuracy_score(y_train, candidate.predict(X_train))
        test_score = accuracy_score(y_test, candidate.predict(X_test))
        malignant_recall = recall_score(
            y_test,
            candidate.predict(X_test),
            pos_label=0
        )

        cv_results.append({
            "n_estimators": n,
            "max_depth": depth,
            "CV Accuracy (mean)": scores.mean(),
            "CV Accuracy (std)": scores.std(),
            "Train Accuracy": train_score,
            "Test Accuracy": test_score,
            "Malignant Recall": malignant_recall
        })

cv_results_df = pd.DataFrame(cv_results)
cv_results_df = cv_results_df.sort_values(
    by="CV Accuracy (mean)",
    ascending=False
)

print(cv_results_df.round(4).to_string(index=False))


In [ ]:
importance = pd.DataFrame({
    "Feature" : X.columns,
    "importance" : model.feature_importances_

})

importance = importance.sort_values(
    by ="importance",
    ascending=False

)
print(importance.head(10))

## 5. Feature Importance Analysis

Feature importance is used to identify which input features contribute most to the Random Forest model's tree-based decisions.

* **model.feature_importances_:** Returns the model's impurity-based importance score for each feature after training.
* **X.columns:** Provides the names of the input features.
* **pd.DataFrame():** Combines feature names and importance scores into one table.
* **sort_values():** Sorts the features according to their importance scores.
* **by="Importance":** Specifies the column used for sorting.
* **ascending=False:** Sorts the scores from highest to lowest.
* **head(10):** Displays the ten features with the highest scores.

The default importance measure in scikit-learn's Random Forest is based on the reduction in node impurity across the trees.

A higher score means that a feature contributed more to impurity reduction within the trained forest, according to this measure. It does not mean that the feature causes the target outcome or directly accounts for the same percentage of prediction accuracy.

Feature importance helps us interpret the model and identify features that may deserve further investigation.


In [ ]:
top_features = importance.head(10).sort_values(by="importance")

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Top 10 Important Features - Random Forest")
plt.tight_layout()
plt.show()


## 6. Visualizing Feature Importance

We create a horizontal bar chart to make the feature importance scores easier to compare.

* **top_features:** Stores the ten most important features.
* **plt.figure(figsize=(10, 6)):** Creates a figure with enough space to display the feature names.
* **plt.barh():** Draws horizontal bars, with feature names on the vertical axis and importance scores on the horizontal axis.
* **xlabel():** Labels the horizontal axis.
* **ylabel():** Labels the vertical axis.
* **title():** Adds a descriptive chart title.
* **tight_layout():** Adjusts the spacing to reduce overlap between chart elements.
* **show():** Displays the chart.

The length of each bar represents the feature's importance score. This visualization helps us quickly identify and compare the features with the highest scores.


## 7. Interpreting Feature Importance and Correlation

Features such as `worst perimeter`, `worst area`, `worst radius`, `mean perimeter`, `mean radius`, and `mean area` describe related physical properties of cell nuclei: size and boundary measurements. The exact ranking should be read from the table and chart above, because importance scores can vary with the train/test split and model settings.

Radius, perimeter, and area tend to cluster together because they are strongly correlated: they all describe nuclear size from different but related measurements. When correlated features contain overlapping information, a Random Forest can split importance among them. As a result, one of these features may rank lower even though the group collectively carries useful information.

Permutation importance provides a complementary view. It measures how much the model's test-set score changes when one feature is shuffled. A larger decrease in accuracy suggests the fitted model relies more on that feature for test predictions. With correlated features, permutation importance can be low for one feature because the model can still use a correlated alternative. The two rankings therefore may differ.

These are model-interpretation results, not evidence of causation or a medical diagnosis.


In [ ]:
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    scoring="accuracy"
)

permutation_scores = pd.DataFrame({
    "Feature": X.columns,
    "Permutation Importance Mean": perm.importances_mean,
    "Permutation Importance Std": perm.importances_std
}).sort_values(
    by="Permutation Importance Mean",
    ascending=False
)

importance_comparison = importance.merge(
    permutation_scores,
    on="Feature",
    how="left"
).sort_values(
    by="Permutation Importance Mean",
    ascending=False
)

print(importance_comparison.head(10).to_string(index=False))

top_perm = importance_comparison.head(10).sort_values(
    by="Permutation Importance Mean"
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_perm["Feature"],
    top_perm["Permutation Importance Mean"],
    xerr=top_perm["Permutation Importance Std"]
)
plt.xlabel("Decrease in Test Accuracy After Permutation")
plt.ylabel("Feature")
plt.title("Top 10 Features by Permutation Importance")
plt.tight_layout()
plt.show()


## 8. Confusion Matrix and Malignant-Class Recall

The confusion matrix uses rows for true labels and columns for predicted labels, ordered as class 0 (malignant) and class 1 (benign). The upper-right cell counts malignant cases incorrectly predicted as benign. In the notebook's original evaluation, this cell contains **2 cases**; these are especially costly errors in a medical setting because a malignant case is missed.

For this reason, monitor **malignant-class recall**, calculated with `recall_score(y_test, y_prec, pos_label=0)`. Recall measures the proportion of actual malignant cases correctly identified. Overall accuracy alone can hide this kind of error.


In [ ]:
cm = confusion_matrix(y_test, y_prec, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Malignant (0)", "Benign (1)"]
)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Random Forest")
plt.tight_layout()
plt.show()

malignant_recall = recall_score(y_test, y_prec, pos_label=0)
print("Malignant-class recall =", round(malignant_recall, 4))


## 9. Conclusion

In this project, a Random Forest classifier was trained on the Breast Cancer Wisconsin dataset. The model was evaluated using a train/test split, classification metrics, a confusion matrix, and five-fold stratified cross-validation to compare several `n_estimators` and `max_depth` settings.

Impurity-based feature importance and permutation importance provide complementary views of which features the model uses. Correlated measurements such as radius, perimeter, and area can share importance, so individual rankings should be interpreted carefully.

For this dataset, **class 0 represents malignant and class 1 represents benign**. In the original evaluation, two malignant cases were predicted as benign. These are particularly costly errors in a medical screening context, so malignant-class recall should be monitored alongside overall accuracy. This model is an educational experiment, not a clinically validated diagnostic system.
